# 당뇨병 데이터 분석 — 연습용(practice)

scikit-learn 당뇨병 데이터셋을 분석하는 실습.
심화 EDA → 피처 엔지니어링 → 데이터 증강(엄격 비교) → 다중 모델·튜닝 → 모델 해석의 전체 파이프라인 구성.

**실습 방법**: `# TODO` 빈칸(`______`)을 채운 뒤 셀 실행. 막히면 답지용과 비교.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import skew, kurtosis
from IPython.display import display

# 한글 폰트 자동 선택 (Mac/Linux/Win 호환)
for cand in ["AppleGothic", "NanumGothic", "Noto Sans CJK KR", "Malgun Gothic", "NanumBarunGothic"]:
    if any(cand in f.name for f in fm.fontManager.ttflist):
        plt.rcParams["font.family"] = cand
        break
plt.rcParams["axes.unicode_minus"] = False

from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor, IsolationForest)
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (train_test_split, KFold, RepeatedKFold,
                                     cross_val_score, RandomizedSearchCV, learning_curve)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.mixture import GaussianMixture

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_validate
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

RANDOM_STATE = 42
print("준비 완료")

## 1. 데이터 로드 및 품질 점검

결측·중복 점검과 요약 통계로 데이터 상태 파악.

In [ ]:
# [TODO 1] 원본 스케일로 로드 후 sex 를 0/1 로 인코딩 (정규화를 풀어 해석 가능하게)
data = load_diabetes(______)                  # scaled 인자로 정규화 해제 + as_frame 으로 DataFrame 반환
X = data.data.copy(); y = data.target.copy()
X["sex"] = (X["sex"] == ______).astype(int)   # 두 성별 범주 중 한쪽을 True(=1)로 만들어 이진화
df = X.assign(target=y)                        # 피처에 target 열을 합친 분석용 DataFrame
print("형태:", df.shape); display(df.head())

## 2. 심화 탐색적 분석(EDA)

### 2.1 타깃 분포와 정규성

왜도·첨도로 분포의 치우침 정량화.

In [ ]:
# [TODO 2] 타깃 분포와 정규성(왜도·첨도) 확인
print(f"왜도={skew(y):.3f} 첨도={kurtosis(y):.3f}")
px.histogram(______).show()                   # target 분포를 히스토그램 + 상자그림(marginal)으로 표현

### 2.2 성별·나이 분석 (정규화 해제 후 범주형/실수형 복원)

정규화 상태로는 해석 불가했던 sex(범주형)와 age(나이, 세)를 원본 스케일로 복원해 개별 분석. sex는 1/2 → 그룹 A/B 범주로, age는 연령대로 구간화해 타깃과의 관계 확인.

In [ ]:
# [TODO 3] 성별(범주형) EDA — 그룹별 타깃 차이 확인
df_eda = df.copy()
df_eda["성별"] = X["sex"].map(______)          # 0/1 코드를 사람이 읽는 라벨(그룹 A/B)로 변환
display(df_eda.groupby(______)["target"].mean().round(1))   # 성별 그룹별 target 평균 비교
px.box(______).show()                          # 성별에 따른 target 분포를 상자그림으로

In [ ]:
# [TODO 4] 나이 EDA + 연령대 구간화
px.histogram(______).show()                    # 나이(age) 분포 확인
px.scatter(______).show()                      # 나이~target 관계를 추세선(trendline)과 함께, 성별로 색 구분
df_eda["연령대"] = pd.cut(______)              # 연속형 나이를 구간(~30대/40대/50대/60대+)으로 범주화
display(df_eda.groupby(______, observed=True)["target"].mean().round(1))  # 연령대별 target 평균
px.box(______).show()                          # 연령대별 target 분포 상자그림

### 2.3 타깃 구간별 피처 분포

타깃을 사분위로 나눠 피처가 구간별로 어떻게 달라지는지 확인.

In [ ]:
# [TODO 5] 타깃 사분위 구간별 핵심 피처 분포
dfq = df.copy()
dfq["타깃구간"] = pd.qcut(______)              # target 을 4분위로 나눠 구간 라벨 부여
for f in ["bmi", "s5", "bp", "s3"]:
    px.violin(______).show()                   # 각 피처가 타깃 구간별로 어떻게 달라지는지 바이올린으로

### 2.4 상관 구조와 다중공선성

계층 클러스터맵으로 유사 변수 군집 확인 후, VIF로 공선성 진단.

In [ ]:
# [TODO 6] 상관관계 + 계층 클러스터링 (유사 변수 군집 확인)
cg = sns.clustermap(______)                    # df.corr() 을 클러스터링해 비슷한 변수끼리 묶어 표시
cg.fig.suptitle("상관관계 계층 클러스터맵", y=1.02); plt.show()

In [ ]:
# [TODO 7] 다중공선성 진단(VIF) — 클수록 다른 변수로 설명되는 정도가 큼
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
Xc = add_constant(X)                           # 절편(상수항)을 더해야 VIF 계산이 정확
vif = pd.DataFrame({"변수": X.columns,
    "VIF": [variance_inflation_factor(______) for i in range(X.shape[1])]   # 각 피처 i 의 VIF
}).sort_values("VIF", ascending=False)
display(vif.round(2))

혈청 지표(s1·s2 등)는 서로 강하게 연관되어 VIF가 높게 나타남 — 정규화 모델이 유리한 근거.

### 2.5 비선형 의존성(상호정보량)

선형 상관이 못 잡는 비선형 관계를 MI로 보완.

In [ ]:
# [TODO 8] 상호정보량(MI): 선형 상관이 놓치는 비선형 의존성 포착
mi = pd.Series(mutual_info_regression(______), index=X.columns).sort_values(ascending=False)  # 각 피처와 target 의 상호정보량
px.bar(______).show()                          # 변수별 상호정보량을 가로 막대로 시각화

### 2.6 차원 축소 및 이상치 탐지

PCA로 구조를 2D로 압축하고 IsolationForest로 이상치 식별.

In [ ]:
# [TODO 9] PCA 2차원 투영 (구조를 2D로 압축해 시각화)
Xs = StandardScaler().fit_transform(X)         # PCA 전 표준화는 필수
pca = PCA(______).fit(Xs)                       # 주성분 2개로 차원 축소
pcs = pca.transform(Xs)
pdf = pd.DataFrame(pcs, columns=["PC1", "PC2"]); pdf["target"] = y.values
px.scatter(______).show()                      # PC1-PC2 평면에 target 으로 색칠해 산점도

In [ ]:
# [TODO 10] IsolationForest 로 이상치 탐지
iso = IsolationForest(______).fit(Xs)          # 표본의 약 5%를 이상치로 가정(contamination)
flag = iso.predict(Xs)                          # 정상=1, 이상치=-1 로 반환
print("이상치:", int((flag == -1).sum()), "건")
pdf["판정"] = np.where(______)                  # -1 인 표본을 "이상치"로 라벨링
px.scatter(______).show()                      # 이상치를 PCA 평면에서 색으로 구분

## 3. 피처 엔지니어링

EDA에서 영향력이 큰 bmi·s5를 중심으로 상호작용·비선형 파생변수 생성.

In [ ]:
# [TODO 11] 피처 엔지니어링 — EDA 에서 영향이 큰 bmi·s5 로 파생변수 생성
def add_features(d_in):
    d = d_in.copy()
    d["bmi_s5"] = ______        # 핵심 두 변수의 상호작용(곱)
    d["bmi_bp"] = ______        # 비만 × 혈압
    d["s5_bp"]  = ______        # 중성지방 × 혈압
    d["tc_hdl_gap"] = ______    # 총콜레스테롤 - HDL (지질 균형)
    d["bmi_sq"] = ______        # bmi 의 비선형(2차) 항
    return d
Xfe = add_features(X)
print("원본:", X.shape[1], "→ 파생 후:", Xfe.shape[1])

## 4. 모델링: 여러 모델 비교

선형 ~ 부스팅까지 13종 회귀 모델(XGBoost·LightGBM 포함)을 동일한 5-fold 교차검증으로 학습하고 R²·RMSE·MAE 다중 지표로 비교.

In [ ]:
# [TODO 12] 13종 회귀 모델 라인업 + 다중 지표(R²·RMSE·MAE) 비교
def sc(m): return make_pipeline(StandardScaler(), m)   # 선형·거리 모델은 표준화 파이프라인으로
models = {
    "LinearRegression": sc(LinearRegression()),
    "Ridge": sc(Ridge(alpha=1.0)),
    "Lasso": sc(Lasso(alpha=0.1, max_iter=10000)),
    "ElasticNet": sc(ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)),
    "SVR": sc(SVR(C=100, gamma="scale")),
    "KNN": sc(KNeighborsRegressor(n_neighbors=15)),
    "DecisionTree": DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE),
    "RandomForest": RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=400, random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "HistGBM": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
    "XGBoost": XGBRegressor(______),       # 부스팅: 트리를 순차적으로 더해 잔차를 줄임 (verbosity=0 로 로그 끔)
    "LightGBM": LGBMRegressor(______),     # 빠른 부스팅 구현 (verbose=-1 로 로그 끔)
}
scoring = ______                            # 한 번에 측정할 지표 3종(R²·RMSE·MAE) 딕셔너리
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, m in models.items():
    r = cross_validate(______)              # 모델을 교차검증하며 여러 지표를 동시에 측정
    rows.append({"모델": name, "R²": r["test_R2"].mean(), "R² 표준편차": r["test_R2"].std(),
                 "RMSE": -r["test_RMSE"].mean(), "MAE": -r["test_MAE"].mean()})
cmp = pd.DataFrame(rows).sort_values("R²", ascending=False).reset_index(drop=True)
cmp.insert(0, "순위", cmp.index + 1); display(cmp.round(4))

In [ ]:
# [TODO 13] 모델별 성능 비교 시각화
px.bar(______).show()      # R²(높을수록 좋음)을 오차막대와 함께 막대그래프로
px.bar(______).show()      # RMSE(낮을수록 좋음) 기준으로 정렬해 막대그래프로

## 5. 데이터 증강(뻥튀기)과 엄격한 비교

442건은 적은 편이라 학습 데이터 증강을 시도. **핵심 원칙**: 증강은 학습 폴드에만 적용하고 테스트 폴드는 항상 원본 유지 → 데이터 누수 차단. 증강이 실제로 성능을 올리는지 동일 교차검증으로 정직하게 비교.

In [ ]:
# [TODO 14] 데이터 증강 함수 정의 (학습 데이터 뻥튀기)
def aug_gaussian(Xtr, ytr, n_new, noise=0.5, seed=0):
    rng = np.random.RandomState(seed)
    idx = rng.randint(______)            # 기존 표본에서 무작위로 n_new 개 인덱스 선택
    Xn = Xtr.values[idx] + rng.normal(______) * Xtr.values.std(0)   # 표본에 정규분포 잡음을 더해 유사 표본 생성
    yn = ytr.values[idx] + rng.normal(0, noise, n_new) * ytr.values.std()
    return pd.DataFrame(Xn, columns=Xtr.columns), pd.Series(yn)

def aug_gmm(Xtr, ytr, n_new, n_comp=8, seed=0):
    Z = np.column_stack([Xtr.values, ytr.values])    # 피처와 타깃을 함께 분포 학습
    gm = GaussianMixture(______).fit(Z)              # 가우시안 혼합으로 데이터 분포 추정
    samp, _ = gm.sample(______)                      # 추정한 분포에서 새 표본 생성
    return pd.DataFrame(samp[:, :-1], columns=Xtr.columns), pd.Series(samp[:, -1])
print("증강 함수 정의 완료")

In [ ]:
# [TODO 15] 누수 없는 증강 비교 — 증강은 학습 폴드에만, 테스트는 원본 유지
def eval_aug(Xd, aug_fn=None, mult=1.0, seed=RANDOM_STATE):
    cv = KFold(5, shuffle=True, random_state=seed); sc = []
    for tr, te in cv.split(Xd):
        Xtr, Xte = Xd.iloc[tr], Xd.iloc[te]; ytr, yte = y.iloc[tr], y.iloc[te]
        if aug_fn is not None:
            Xa, ya = aug_fn(______)              # 학습 폴드 크기의 mult 배만큼 증강 표본 생성
            Xtr = pd.concat([Xtr, Xa], ignore_index=True); ytr = pd.concat([ytr, ya], ignore_index=True)
        m = HistGradientBoostingRegressor(random_state=RANDOM_STATE); m.fit(Xtr, ytr)
        sc.append(r2_score(______))              # 평가는 항상 '원본' 테스트셋으로 (누수 방지)
    return np.array(sc)
res = {"원본(증강 없음)": eval_aug(Xfe),
       "가우시안 +100%": eval_aug(______),       # 가우시안 잡음으로 학습 데이터를 100% 늘려 평가
       "GMM +100%": eval_aug(Xfe, aug_gmm, 1.0),
       "GMM +300%": eval_aug(Xfe, aug_gmm, 3.0)}
aug_df = pd.DataFrame([{"증강 방식": k, "R² 평균": v.mean(), "R² 표준편차": v.std()} for k, v in res.items()])
display(aug_df.round(4))

> 해석 주의: 표 형식 회귀에서 합성 증강은 분포를 모방할 뿐 새로운 정보를 만들지 못해, 성능이 크게 오르지 않거나 오히려 소폭 하락하기도 함. 증강은 만능이 아니며 검증으로 확인하는 자세가 중요.

## 6. 최종 모델 선정 및 튜닝

비교 결과 R² 최고 모델을 최종 선정하고, 트리·부스팅 계열이면 RandomizedSearchCV 로 튜닝.

In [ ]:
# [TODO 16] 최종 모델 선정 → 트리·부스팅이면 RandomizedSearchCV 튜닝
PARAM_DISTS = {
    "RandomForest": {"n_estimators": [200, 400, 600], "max_depth": [None, 4, 6, 8], "min_samples_leaf": [1, 2, 5]},
    "GradientBoosting": {"n_estimators": [100, 200, 300], "learning_rate": [0.02, 0.05, 0.1], "max_depth": [2, 3, 4]},
    "HistGBM": {"learning_rate": [0.02, 0.05, 0.1, 0.2], "max_leaf_nodes": [15, 31, 63], "l2_regularization": [0.0, 0.1, 1.0]},
    "XGBoost": {"n_estimators": [200, 400, 600], "learning_rate": [0.02, 0.05, 0.1], "max_depth": [2, 3, 4]},
    "LightGBM": {"n_estimators": [200, 400, 600], "learning_rate": [0.02, 0.05, 0.1], "num_leaves": [15, 31, 63]},
}
best_name = cmp.iloc[0][______]            # 비교표에서 R² 1위 모델 이름
base = models[best_name]
if best_name in PARAM_DISTS:
    search = RandomizedSearchCV(______).fit(Xfe, y)   # 후보 파라미터를 무작위 탐색해 튜닝
    best = search.best_estimator_; print(f"최종 선정: {best_name} (튜닝됨)")
else:
    best = base; print(f"최종 선정: {best_name} | 선형 계열 기본값 사용")

## 7. 모델 해석

### 7.1 순열 중요도

In [ ]:
# [TODO 17] 최종 모델 홀드아웃 평가 + 순열 중요도
Xtr, Xte, ytr, yte = train_test_split(______)   # 80/20 으로 분할 (random_state 고정)
best.fit(Xtr, ytr); pred = best.predict(Xte)
print(f"홀드아웃 R²={r2_score(yte, pred):.4f} | RMSE={mean_squared_error(yte, pred)**0.5:.2f} | MAE={mean_absolute_error(yte, pred):.2f}")
pi = permutation_importance(______)              # 각 피처를 섞었을 때의 성능 저하로 중요도 측정
pis = pd.Series(pi.importances_mean, index=Xfe.columns).sort_values()
px.bar(______).show()                            # 변수별 순열 중요도를 가로 막대로

### 7.2 잔차 분석

In [ ]:
# [TODO 18] 잔차 분석 (예측값 대 잔차 — 패턴이 없어야 이상적)
resid = ______                                  # 실제값 - 예측값
fig = px.scatter(______)                         # 예측값(x) 대 잔차(y) 산점도
fig.add_hline(y=0, line_dash="dash", line_color="red"); fig.show()

### 7.3 부분의존도(PDP)

In [ ]:
# [TODO 19] 부분의존도(PDP) — 핵심 변수의 한계 효과
top3 = pis.sort_values(ascending=False).index[:3].tolist()   # 중요도 상위 3개 변수
fig, ax = plt.subplots(figsize=(13, 4))
PartialDependenceDisplay.from_estimator(______)  # 최종 모델 기준 상위 변수의 PDP 그리기
plt.tight_layout(); plt.show()

### 7.4 학습곡선

In [ ]:
# [TODO 20] 학습곡선 — 표본 수가 늘수록 성능이 오르는지
sizes, tr_sc, te_sc = learning_curve(______)     # 표본 크기를 늘려가며 학습·검증 R² 측정
fig = go.Figure()
fig.add_scatter(x=sizes, y=tr_sc.mean(1), name="학습 R²", mode="lines+markers")
fig.add_scatter(x=sizes, y=te_sc.mean(1), name="검증 R²", mode="lines+markers")
fig.update_layout(title="학습곡선", xaxis_title="학습 표본 수", yaxis_title="R²"); fig.show()

## 8. 결론 및 시사점

- 정규화를 풀어 age(나이)·sex(성별)를 해석 가능한 범주/실수로 복원 → 성별·연령대별 타깃 차이를 직접 확인
- bmi와 s5가 선형·비선형·중요도 분석 전반에서 일관되게 핵심 인자로 확인됨
- 혈청 지표 간 강한 공선성 존재 → 정규화 선형 모델 또는 트리 계열이 안정적
- 파생변수(상호작용·비선형 항)는 모델에 따라 소폭의 성능 향상 기여
- 데이터 증강은 누수 없는 비교에서 뚜렷한 개선을 주지 못함 → 표 형식 회귀에서 합성 증강의 한계 확인
- 학습곡선의 검증 성능이 평탄 → 표본 수보다 피처 정보량이 성능의 병목
- 실무 결론: 무리한 증강보다 양질의 피처 확보와 적절한 정규화·튜닝이 우선